# Rented-GPU condition, second session (v2 notebook, prebuilt llama.cpp binary)

Team KaiyrbekulyAitbek_TurganbekNurhan, AITU SE-2424. Repository: https://github.com/Newterios/rmt-llm-inference-benchmark

Differences from `colab_benchmark.ipynb` (first session, Tesla T4, 23.10.2026): the llama.cpp server is the **prebuilt release
binary b11853 (ubuntu, CUDA 12.8)** instead of a 54-minute source build, the server listens on port 8090, and the condition name is
chosen from the GPU that Colab actually assigns (`rented_colab_l4` or `rented_colab_t4_run2`). Everything else (model file and checksum,
prompt sets, decoding parameters, 16 x 1,024-token slots, concurrency levels, repeats) is identical to the Mac, the PC and the first session.
This notebook is saved with its outputs after the run (File > Download .ipynb) and committed to the repository as the executed record.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,power.limit --format=csv
import datetime, platform, subprocess
GPU = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
COND = "rented_colab_l4" if "L4" in GPU else "rented_colab_t4_run2"
print(datetime.datetime.now(), platform.platform()); print("GPU:", GPU, "-> condition:", COND)

In [ ]:
%cd /content
!rm -rf rmt-llm-inference-benchmark && git clone -q https://github.com/Newterios/rmt-llm-inference-benchmark.git
%cd /content/rmt-llm-inference-benchmark
!pip install -q httpx pandas numpy tqdm psutil python-dotenv anthropic matplotlib
!git log --oneline | head -3

In [ ]:
# Prebuilt llama.cpp release binary (no source build). Records the build number.
!mkdir -p /content/llama && cd /content/llama && curl -L -s -o llama.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b11853/llama-b11853-bin-ubuntu-cuda-12.8-x64.tar.gz && curl -L -s -o cudart.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b11853/cudart-llama-b11853-bin-ubuntu-cuda-12.8-x64.tar.gz && tar xzf llama.tar.gz && tar xzf cudart.tar.gz && ls
import glob, os
SERVER = [p for p in glob.glob("/content/llama/**/llama-server", recursive=True)][0]
LIBDIR = os.path.dirname(SERVER)
os.environ["LD_LIBRARY_PATH"] = LIBDIR + ":" + "/content/llama" + ":" + os.environ.get("LD_LIBRARY_PATH","")
print(SERVER); print(subprocess.run([SERVER,"--version"],capture_output=True,text=True,env=os.environ).stderr.strip()[:200])
os.makedirs(f"runs/{COND}", exist_ok=True); open(f"runs/{COND}/build_info.txt","w").write("llama.cpp release b11853, llama-b11853-bin-ubuntu-cuda-12.8-x64 + cudart, prebuilt\n")

In [ ]:
!mkdir -p models && curl -L -s -o models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf
!ls -la models/ && sha256sum models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf

In [ ]:
import subprocess, time, httpx, os
server = subprocess.Popen([SERVER, "-m", "models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf", "--alias", "llama-3.1-8b-instruct-q4_k_m",
    "--host", "127.0.0.1", "--port", "8090", "-c", "16384", "-np", "16", "-b", "2048", "-ub", "512", "-ngl", "99", "-fa", "on",
    "--no-cache-prompt", "--metrics", "--reasoning", "off"], stdout=open(f"runs/{COND}/server.log", "w"), stderr=subprocess.STDOUT, env=os.environ)
for _ in range(150):
    try:
        if httpx.get("http://127.0.0.1:8090/health", timeout=2).status_code == 200: break
    except Exception: pass
    if server.poll() is not None: print(open(f"runs/{COND}/server.log").read()[-3000:]); raise SystemExit("llama-server exited")
    time.sleep(2)
props = httpx.get("http://127.0.0.1:8090/props").json()
print({k: props.get(k) for k in ("model_path","total_slots","build_info")}, "n_ctx per slot:", props.get("default_generation_settings",{}).get("n_ctx"))
stamp = time.strftime("%Y%m%d-%H%M%S")
power = subprocess.Popen(f"nvidia-smi --query-gpu=timestamp,name,power.draw,utilization.gpu,memory.used,temperature.gpu --format=csv -l 1 > runs/{COND}/power_{stamp}.csv", shell=True)
!grep -E "offloaded|n_slots|listening|prompt cache" runs/$COND/server.log

In [ ]:
!python bench/run.py --condition $COND --task throughput --set main --concurrency 1 4 16 --repeats 3 --note "Colab second session, v2 notebook"
!python bench/run.py --condition $COND --task quality    --set main --concurrency 4 --repeats 2 --note "Colab second session, v2 notebook"
!nvidia-smi --query-gpu=name,power.draw,memory.used,utilization.gpu --format=csv

In [ ]:
!python bench/run.py --condition $COND --task throughput --set pilot --concurrency 1 4 16 --repeats 2 --note "Colab second session, v2 notebook"
!python bench/run.py --condition $COND --task quality    --set pilot --concurrency 4 --repeats 1 --note "Colab second session, v2 notebook"


In [ ]:
import subprocess; subprocess.run("pkill -f nvidia-smi; pkill -f llama-server", shell=True)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > runs/$COND/gpu_info.txt; cat runs/$COND/gpu_info.txt
!cd runs && zip -q -r /content/{COND}_runs.zip $COND && ls -la /content/{COND}_runs.zip
from google.colab import files; files.download(f"/content/{COND}_runs.zip")

After the download: File > Download > Download .ipynb, then commit the executed notebook and the unzipped `runs/<condition>/` folder to the repository.